# Библиотеки

In [ ]:
!pip install numpy==1.21.6 scikit-learn==1.0.2 tensorboard==2.9.0 torch==1.12.1 tqdm==4.64.0 transformers==4.21.1

In [ ]:
import os
import random
from collections import Counter, defaultdict, namedtuple
from typing import Tuple, List, Dict, Any

import torch
import numpy as np

from tqdm import tqdm, trange

# Зафиксируем seed для воспроизводимости результатов

In [ ]:
def set_global_seed(seed: int) -> None:
    """
    Set global seed for reproducibility.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_global_seed(42)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Подготовка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def read_conll2003(path: str, lower: bool = True) -> Tuple[List[List[str]], List[List[str]]]:
    """
    Prepare data in CoNNL like format.

    Args:
        path (str): Path to file in CoNLL-2003 format
        lower (bool): Whether to convert text to lowercase (default True)

    Returns:
        Tuple[List[List[str]], List[List[str]]]: Tuple containing two lists:
            - First list contains token sequences
            - Second list contains corresponding labels
    """
    token_seq = []
    label_seq = []
    print(f"Начинаю чтение файла: {path}")


    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл {path} не найден")

    current_tokens = []
    current_labels = []

    try:
        with open(path, 'r', encoding='utf-8') as file:
            line_num = 0
            for line in file:
                line_num += 1
                line = line.strip()
                print(f"Строка {line_num}: '{line}'")


                if not line or line.startswith('#'):
                    print("Найден разделитель последовательности")
                    if current_tokens:
                        print(f"Сохраняем последовательность: {len(current_tokens)} токенов")
                        token_seq.append(current_tokens)
                        label_seq.append(current_labels)
                        current_tokens = []
                        current_labels = []
                    continue


                if line.startswith('-DOCSTART-'):
                    print("Пропускаем строку -DOCSTART-")
                    continue


                columns = line.split()
                print(f"Колонки после разделения: {columns}")


                if len(columns) >= 2:
                    word = columns[0]
                    label = columns[-1]

                    print(f"Слово: '{word}', Метка: '{label}'")


                    if lower:
                        word = word.lower()

                    current_tokens.append(word)
                    current_labels.append(label)


            if current_tokens:
                print(f"Сохраняем последнюю последовательность: {len(current_tokens)} токенов")
                token_seq.append(current_tokens)
                label_seq.append(current_labels)

        print(f"\nРезультат:")
        print(f"Количество последовательностей: {len(token_seq)}")
        if token_seq:
            print(f"Длина первой последовательности: {len(token_seq[0])}")
        else:
            print("Списки пустые")

    except Exception as e:
        print(f"Произошла ошибка: {str(e)}")
        raise

    return token_seq, label_seq

In [2]:
!git clone https://github.com/google/NeuroNER-CSPMC.git

Cloning into 'NeuroNER-CSPMC'...
remote: Enumerating objects: 698, done.
remote: Total 698 (delta 0), reused 0 (delta 0), pack-reused 698 (from 1)
Receiving objects: 100% (698/698), 114.56 MiB | 15.75 MiB/s, done.
Resolving deltas: 100% (470/470), done.


In [3]:
ls -a NeuroNER-CSPMC/neuroner/data/conll2003/en

./  ../  metadata  test.txt  train.txt  valid.txt


In [ ]:
train_token_seq, train_label_seq = read_conll2003('NeuroNER-CSPMC/neuroner/data/conll2003/en/train.txt')
valid_token_seq, valid_label_seq = read_conll2003('NeuroNER-CSPMC/neuroner/data/conll2003/en/valid.txt')
test_token_seq, test_label_seq = read_conll2003('NeuroNER-CSPMC/neuroner/data/conll2003/en/test.txt')

Выходные данные были обрезаны до нескольких последних строк (5000).
Слово: 'games', Метка: 'O'
Строка 48680: 'on IN B-PP O'
Колонки после разделения: ['on', 'IN', 'B-PP', 'O']
Слово: 'on', Метка: 'O'
Строка 48681: 'Friday NNP B-NP O'
Колонки после разделения: ['Friday', 'NNP', 'B-NP', 'O']
Слово: 'Friday', Метка: 'O'
Строка 48682: '( ( O O'
Колонки после разделения: ['(', '(', 'O', 'O']
Слово: '(', Метка: 'O'
Строка 48683: 'home NN B-NP O'
Колонки после разделения: ['home', 'NN', 'B-NP', 'O']
Слово: 'home', Метка: 'O'
Строка 48684: 'team NN I-NP O'
Колонки после разделения: ['team', 'NN', 'I-NP', 'O']
Слово: 'team', Метка: 'O'
Строка 48685: 'in IN B-PP O'
Колонки после разделения: ['in', 'IN', 'B-PP', 'O']
Слово: 'in', Метка: 'O'
Строка 48686: 'CAPS NNP B-NP O'
Колонки после разделения: ['CAPS', 'NNP', 'B-NP', 'O']
Слово: 'CAPS', Метка: 'O'
Строка 48687: ') ) O O'
Колонки после разделения: [')', ')', 'O', 'O']
Слово: ')', Метка: 'O'
Строка 48688: ': : O O'
Колонки после разделения: [':

In [ ]:
for token, label in zip(train_token_seq[0], train_label_seq[0]):
    print(f"{token}\t{label}")

eu	B-ORG
rejects	O
german	B-MISC
call	O
to	O
boycott	O
british	B-MISC
lamb	O
.	O


In [ ]:
for token, label in zip(valid_token_seq[0], valid_label_seq[0]):
    print(f"{token}\t{label}")

cricket	O
-	O
leicestershire	B-ORG
take	O
over	O
at	O
top	O
after	O
innings	O
victory	O
.	O


In [ ]:
for token, label in zip(test_token_seq[0], test_label_seq[0]):
    print(f"{token}\t{label}")

soccer	O
-	O
japan	B-LOC
get	O
lucky	O
win	O
,	O
china	B-PER
in	O
surprise	O
defeat	O
.	O


# Подготовка словарей
Реализуем функции get_token2idx и get_label2idx

In [ ]:
token2cnt = Counter([token for sentence in train_token_seq for token in sentence])

In [ ]:
token2cnt.most_common(10)

[('the', 8390),
 ('.', 7374),
 (',', 7290),
 ('of', 3815),
 ('in', 3621),
 ('to', 3424),
 ('a', 3199),
 ('and', 2872),
 ('(', 2861),
 (')', 2861)]

In [ ]:
print(f"Количество уникальных слов в тренировочном датасете: {len(token2cnt)}")
print(f"Количество слов встречающихся только один раз в тренировочном датасете: {len([token for token, cnt in token2cnt.items() if cnt == 1])}")

Количество уникальных слов в тренировочном датасете: 21009
Количество слов встречающихся только один раз в тренировочном датасете: 10060


In [ ]:
# используйте параметр min_count для того, чтобы отсекать слова частотой cnt < min_count

def get_token2idx(
    token2cnt: Dict[str, int],
    min_count: int,
) -> Dict[str, int]:
    """
    Get mapping from tokens to indices to use with Embedding layer.

    Args:
        token2cnt (Dict[str, int]): Dictionary mapping tokens to their frequencies
        min_count (int): Minimum frequency threshold for tokens

    Returns:
        Dict[str, int]: Dictionary mapping tokens to indices
    """
    token2idx: Dict[str, int] = {}

    # Сначала добавляем специальные токены
    special_tokens = ['<PAD>', '<UNK>']
    for idx, token in enumerate(special_tokens):
        token2idx[token] = idx

    # Отсекаем слова с частотой меньше min_count
    # Пример: если min_count = 2, слова с частотой 1 будут отброшены
    filtered_tokens = [
        token for token, count in token2cnt.items()
        if count >= min_count
    ]

    # Добавляем остальные слова, начиная с индекса после специальных токенов
    for idx, token in enumerate(filtered_tokens, len(special_tokens)):
        token2idx[token] = idx

    return token2idx

In [ ]:
token2idx = get_token2idx(token2cnt, min_count=2)

In [ ]:
# Функция для сортировки тегов, чтобы сначала был тег O, потом теги B- и только после теги I- (можно задать вручную)

def sort_labels_func(x: str) -> int:
    if x == "O":
        return 0
    elif x.startswith("B-"):
        return 1
    else:
        return 2

label_set = sorted(
    set(label for sentence in train_label_seq for label in sentence),
    key=lambda x: (sort_labels_func(x), x),
)

In [ ]:
label_set

['O', 'B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER']

In [ ]:
def get_label2idx(label_set: List[str]) -> Dict[str, int]:
    """
    Get mapping from labels to indices.

    Args:
        label_set (List[str]): List of unique labels

    Returns:
        Dict[str, int]: Dictionary mapping labels to indices
    """
    label2idx: Dict[str, int] = {}

    # Создаем словарь соответствия меток и индексов
    for idx, label in enumerate(label_set):
        label2idx[label] = idx

    return label2idx

In [ ]:
label2idx = get_label2idx(label_set)

In [ ]:
label2idx

{'O': 0,
 'B-LOC': 1,
 'B-MISC': 2,
 'B-ORG': 3,
 'B-PER': 4,
 'I-LOC': 5,
 'I-MISC': 6,
 'I-ORG': 7,
 'I-PER': 8}

In [ ]:
for token, idx in list(token2idx.items())[:10]:
    print(f"{token}\t{idx}")

<PAD>	0
<UNK>	1
eu	2
german	3
call	4
to	5
boycott	6
british	7
lamb	8
.	9


In [ ]:
for label, idx in label2idx.items():
    print(f"{label}\t{idx}")

O	0
B-LOC	1
B-MISC	2
B-ORG	3
B-PER	4
I-LOC	5
I-MISC	6
I-ORG	7
I-PER	8


In [ ]:
len(get_token2idx(token2cnt, min_count=1))

21011

# Подготовка датасета и загрузчика

In [ ]:
from torch.utils.data import Dataset

class NERDataset(Dataset):
    """
    PyTorch Dataset for NER.
    """
    def __init__(
        self,
        token_seq: List[List[str]],
        label_seq: List[List[str]],
        token2idx: Dict[str, int],
        label2idx: Dict[str, int],
    ):
        """
        Args:
            token_seq: List of token sequences
            label_seq: List of label sequences
            token2idx: Dictionary mapping tokens to indices
            label2idx: Dictionary mapping labels to indices
        """
        self.token2idx = token2idx
        self.label2idx = label2idx
        self.token_seq = [self.process_tokens(tokens, token2idx) for tokens in token_seq]
        self.label_seq = [self.process_labels(labels, label2idx) for labels in label_seq]

    def __len__(self):
        return len(self.token_seq)

    def __getitem__(
        self,
        idx: int,
    ) -> Tuple[torch.LongTensor, torch.LongTensor]:
        """
        Args:
            idx: Index of the sample

        Returns:
            Tuple of two LongTensors:
            - Token indices tensor
            - Label indices tensor
        """
        return torch.LongTensor(self.token_seq[idx]), torch.LongTensor(self.label_seq[idx])

    @staticmethod
    def process_tokens(
        tokens: List[str],
        token2idx: Dict[str, int],
        unk: str = "<UNK>",
    ) -> List[int]:
        """
        Transform list of tokens into list of tokens' indices.

        Args:
            tokens: List of tokens
            token2idx: Dictionary mapping tokens to indices
            unk: Unknown token symbol

        Returns:
            List of token indices
        """
        return [token2idx.get(token, token2idx[unk]) for token in tokens]

    @staticmethod
    def process_labels(
        labels: List[str],
        label2idx: Dict[str, int],
    ) -> List[int]:
        """
        Transform list of labels into list of labels' indices.

        Args:
            labels: List of labels
            label2idx: Dictionary mapping labels to indices

        Returns:
            List of label indices
        """
        return [label2idx[label] for label in labels]

In [ ]:
train_dataset = NERDataset(
    token_seq=train_token_seq,
    label_seq=train_label_seq,
    token2idx=token2idx,
    label2idx=label2idx,
)
valid_dataset = NERDataset(
    token_seq=valid_token_seq,
    label_seq=valid_label_seq,
    token2idx=token2idx,
    label2idx=label2idx,
)
test_dataset = NERDataset(
    token_seq=test_token_seq,
    label_seq=test_label_seq,
    token2idx=token2idx,
    label2idx=label2idx,
)

In [ ]:
train_dataset[0]

(tensor([2, 1, 3, 4, 5, 6, 7, 8, 9]), tensor([3, 0, 2, 0, 0, 0, 2, 0, 0]))

In [ ]:
valid_dataset[0]

(tensor([1736,  570, 1776,  197,  686,  145,  348,  111, 1818, 1557,    9]),
 tensor([0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0]))

In [ ]:
test_dataset[0]

(tensor([1515,  570, 1433, 1728, 4892, 2013,   67,  309,  215, 3156, 3138,    9]),
 tensor([0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0]))

##  NERCollator

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class NERCollator:
    """
    Collator that handles variable-size sentences.
    """
    def __init__(
        self,
        token_padding_value: int,
        label_padding_value: int,
    ):
        """
        Args:
            token_padding_value: Value to use for padding token sequences
            label_padding_value: Value to use for padding label sequences
        """
        self.token_padding_value = token_padding_value
        self.label_padding_value = label_padding_value

    def __call__(
        self,
        batch: List[Tuple[torch.LongTensor, torch.LongTensor]],
    ) -> Tuple[torch.LongTensor, torch.LongTensor]:
        """
        Args:
            batch: List of tuples (tokens, labels) from dataset

        Returns:
            Tuple of padded tensors:
            - Padded token tensor
            - Padded label tensor
        """
        tokens, labels = zip(*batch)
        tokens = pad_sequence(tokens, padding_value=self.token_padding_value, batch_first=True)
        labels = pad_sequence(labels, padding_value=self.label_padding_value, batch_first=True)

        return tokens, labels

In [ ]:
collator = NERCollator(
    token_padding_value=token2idx["<PAD>"],
    label_padding_value=-1,
)

In [ ]:
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)
valid_dataloader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=1,  # для корректных замеров метрик оставить batch_size=1
    shuffle=False, # для корректных замеров метрик оставить shuffle=False
    collate_fn=collator,
)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,  # для корректных замеров метрик оставить batch_size=1
    shuffle=False, # для корректных замеров метрик оставить shuffle=False
    collate_fn=collator,
)

In [ ]:
tokens, labels = next(iter(train_dataloader))

tokens = tokens.to(device)
labels = labels.to(device)

In [ ]:
tokens

tensor([[1515,  570, 1024,   80, 2288, 2254, 2045,  215, 5259,    9],
        [6873,  404, 6874, 1686, 1341, 1679, 1679,  582,  236, 1679]],
       device='cuda:0')

In [ ]:
labels

tensor([[0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
        [3, 7, 7, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')

# BiLSTM model

In [ ]:
class BiLSTM(torch.nn.Module):
    """
    Bidirectional LSTM architecture.
    """

    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool,
        n_classes: int,
    ):
        super().__init__()

        self.embedding = torch.nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim) #, padding_idx=-1)
        self.rnn = torch.nn.LSTM(
                                  input_size=embedding_dim,
                                  hidden_size=hidden_size,
                                  num_layers=num_layers,
                                  dropout=dropout if num_layers > 1 else 0.0,
                                  bidirectional=bidirectional)
        # self.attn = None
        self.head = torch.nn.Linear(
                                  in_features=2*hidden_size if bidirectional else hidden_size,
                                  out_features=n_classes)

    def forward(self, tokens: torch.LongTensor) -> torch.Tensor:
        embed = self.embedding(tokens)

        # используем специальную функцию pack_padded_sequence для того, чтобы получить структуру PackedSequence
        # которая не учитывать паддинг при проходе rnn
        length = (tokens != 0).sum(dim=1).detach().cpu()
        packed_embed = torch.nn.utils.rnn.pack_padded_sequence(
            embed, length, batch_first=True, enforce_sorted=False
          )

        # используем специальную функцию pad_packed_sequence для того, чтобы получить тензор из PackedSequence
        packed_rnn_output, _ = self.rnn(packed_embed)
        rnn_output, _ = torch.nn.utils.rnn.pad_packed_sequence(
            packed_rnn_output, batch_first=True)


        logits = self.head(rnn_output)
        return logits.transpose(1, 2)

In [ ]:
model = BiLSTM(
    num_embeddings=len(token2idx),
    embedding_dim=100,
    hidden_size=100,
    num_layers=1,
    dropout=0.3,
    bidirectional=True,
    n_classes=len(label2idx),
).to(device)

In [ ]:
model

BiLSTM(
  (embedding): Embedding(10951, 100)
  (rnn): LSTM(100, 100, bidirectional=True)
  (head): Linear(in_features=200, out_features=9, bias=True)
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)

In [ ]:
outputs = model(tokens)

In [ ]:
outputs

tensor([[[ 0.0292,  0.0292,  0.0292,  0.0292,  0.0292,  0.0292,  0.0292,
           0.0292,  0.0292,  0.0292,  0.0292,  0.0292,  0.0292],
         [ 0.0627,  0.0627,  0.0627,  0.0627,  0.0627,  0.0627,  0.0627,
           0.0627,  0.0627,  0.0627,  0.0627,  0.0627,  0.0627],
         [ 0.0371,  0.0371,  0.0371,  0.0371,  0.0371,  0.0371,  0.0371,
           0.0371,  0.0371,  0.0371,  0.0371,  0.0371,  0.0371],
         [ 0.0256,  0.0256,  0.0256,  0.0256,  0.0256,  0.0256,  0.0256,
           0.0256,  0.0256,  0.0256,  0.0256,  0.0256,  0.0256],
         [ 0.0656,  0.0656,  0.0656,  0.0656,  0.0656,  0.0656,  0.0656,
           0.0656,  0.0656,  0.0656,  0.0656,  0.0656,  0.0656],
         [ 0.0436,  0.0436,  0.0436,  0.0436,  0.0436,  0.0436,  0.0436,
           0.0436,  0.0436,  0.0436,  0.0436,  0.0436,  0.0436],
         [ 0.0727,  0.0727,  0.0727,  0.0727,  0.0727,  0.0727,  0.0727,
           0.0727,  0.0727,  0.0727,  0.0727,  0.0727,  0.0727],
         [-0.0191, -0.0191, -0.019

In [ ]:
outputs.shape

torch.Size([2, 9, 13])

In [ ]:
outputs.transpose(1,2).size()

torch.Size([2, 13, 9])

# Эксперименты (BiLSTM)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

IGNORE_INDEX = -100

def compute_metrics(
    outputs: torch.Tensor,   # [B, T, C]
    labels: torch.LongTensor, # [B, T] (PAD = -1)
    ignore_index: int = IGNORE_INDEX,
) -> Dict[str, float]:
    metrics = {}

    # if outputs.size(1) != labels.size(1):
    #     T = min(outputs.size(1), labels.size(1))
    #     outputs = outputs[:, :T, :]
    #     labels  = labels[:,  :T]
    # print("OUTPUTS", outputs.shape)

    # (B T C ::  B S Ed) -> preds(B T LabelProb) (1 60 10) - argmax(dim=-1) -> (1 60 1)
    # (1 10 60) - argmax(dim=-1) -> (1 10 1)
    preds = outputs.argmax(dim=-1)
    mask = (labels != -1)
    y_true = labels[mask].detach().cpu().numpy()
    y_pred = preds[mask].detach().cpu().numpy()

    metrics["accuracy"]            = accuracy_score(y_true, y_pred)
    metrics["precision_micro"]     = precision_score(y_true, y_pred, average="micro",    zero_division=0)
    metrics["precision_macro"]     = precision_score(y_true, y_pred, average="macro",    zero_division=0)
    metrics["precision_weighted"]  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    metrics["recall_micro"]        = recall_score(y_true, y_pred,  average="micro",      zero_division=0)
    metrics["recall_macro"]        = recall_score(y_true, y_pred,  average="macro",      zero_division=0)
    metrics["recall_weighted"]     = recall_score(y_true, y_pred,  average="weighted",   zero_division=0)
    metrics["f1_micro"]            = f1_score(y_true, y_pred,      average="micro",      zero_division=0)
    metrics["f1_macro"]            = f1_score(y_true, y_pred,      average="macro",      zero_division=0)
    metrics["f1_weighted"]         = f1_score(y_true, y_pred,      average="weighted",   zero_division=0)

    return metrics


## Обучения и тестирования train_epoch и evaluate_epoch

In [ ]:
def _move_batch_to_device(tokens, labels, device, transformers: bool):
    """Move batch to device; для BiLSTM берём именно idxы токенов."""
    if transformers:
        tokens = {k: v.to(device) for k, v in tokens.items()}
    else:
        if isinstance(tokens, dict):        # если пришёл BatchEncoding
            tokens = tokens["input_ids"]    # BiLSTM ждёт [B, T] индексы
        tokens = tokens.to(device)
    labels = labels.to(device).long()
    return tokens, labels



def _to_btc_for_metrics(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Приводит logits к [B, T, C] (для метрик).
    Допускает вход [B,T,C] или [B,C,T]; сверяет с T = labels.size(1).
    """
    T = labels.size(1)
    if logits.dim() != 3:
        raise ValueError(f"Expected 3D logits, got {logits.shape}")
    if logits.shape[1] == T:         # [B, T, C]
        return logits
    if logits.shape[2] == T:         # [B, C, T]
        return logits.transpose(1, 2)
    raise ValueError(f"Token length T={T} not found in logits shape {logits.shape}")

def _to_bct_for_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Приводит logits к [B, C, T] (для CrossEntropyLoss).
    """
    T = labels.size(1)
    if logits.dim() != 3:
        raise ValueError(f"Expected 3D logits, got {logits.shape}")
    if logits.shape[2] == T:         # [B, C, T]
        return logits
    if logits.shape[1] == T:         # [B, T, C]
        return logits.transpose(1, 2)
    raise ValueError(f"Token length T={T} not found in logits shape {logits.shape}")


def train_epoch(
    model, dataloader, optimizer, criterion, writer, device, epoch, transformers: bool = False
) -> None:
    model.train()
    epoch_loss = []
    batch_metrics_list = defaultdict(list)

    for i, (tokens, labels) in tqdm(enumerate(dataloader), total=len(dataloader), desc="loop over train batches"):
        optimizer.zero_grad()
        tokens, labels = _move_batch_to_device(tokens, labels, device, transformers)

        if transformers:
            out = model(**tokens, labels=labels)     # у HF лосс уже со встроенным ignore_index=-100
            loss = out.loss
            logits_for_metrics = _to_btc_for_metrics(out.logits, labels)   # [B,T,C]
        else:
            logits = model(tokens)                   # [B,T,C] или [B,C,T]
            logits_for_loss = _to_bct_for_loss(logits, labels)             # [B,C,T]
            logits_for_metrics = _to_btc_for_metrics(logits, labels)       # [B,T,C]
            loss = criterion(logits_for_loss, labels)

        loss.backward()
        optimizer.step()

        epoch_loss.append(loss.item())
        writer.add_scalar("batch loss / train", loss.item(), epoch * len(dataloader) + i)

        with torch.no_grad():
            batch_metrics = compute_metrics(outputs=logits_for_metrics, labels=labels, ignore_index=IGNORE_INDEX)

        for name, val in batch_metrics.items():
            batch_metrics_list[name].append(val)
            writer.add_scalar(f"batch {name} / train", val, epoch * len(dataloader) + i)

    avg_loss = float(np.mean(epoch_loss)) if epoch_loss else 0.0
    print(f"Train loss: {avg_loss}\n")
    writer.add_scalar("loss / train", avg_loss, epoch)

    for name, vals in batch_metrics_list.items():
        m = float(np.mean(vals)) if vals else 0.0
        print(f"Train {name}: {m}\n")
        writer.add_scalar(f"{name} / train", m, epoch)


In [ ]:
label2idx

{'O': 0,
 'B-LOC': 1,
 'B-MISC': 2,
 'B-ORG': 3,
 'B-PER': 4,
 'I-LOC': 5,
 'I-MISC': 6,
 'I-ORG': 7,
 'I-PER': 8}

In [ ]:
def evaluate_epoch(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    criterion: torch.nn.Module,
    writer: SummaryWriter,
    device: torch.device,
    epoch: int,
    transformers: bool = False,
) -> None:
    model.eval()
    epoch_loss = []
    batch_metrics_list = defaultdict(list)

    with torch.no_grad():
        for i, (tokens, labels) in tqdm(enumerate(dataloader), total=len(dataloader), desc="loop over test batches"):
            tokens, labels = _move_batch_to_device(tokens, labels, device, transformers)

            if transformers:
                out = model(**tokens, labels=labels)
                loss = out.loss
                logits_for_metrics = _to_btc_for_metrics(out.logits, labels)
            else:
                logits = model(tokens)
                logits_for_loss = _to_bct_for_loss(logits, labels)
                logits_for_metrics = _to_btc_for_metrics(logits, labels)
                loss = criterion(logits_for_loss, labels)

            epoch_loss.append(loss.item())
            writer.add_scalar("batch loss / test", loss.item(), epoch * len(dataloader) + i)

            batch_metrics = compute_metrics(outputs=logits_for_metrics, labels=labels, ignore_index=IGNORE_INDEX)
            for name, val in batch_metrics.items():
                batch_metrics_list[name].append(val)
                writer.add_scalar(f"batch {name} / test", val, epoch * len(dataloader) + i)

    avg_loss = float(np.mean(epoch_loss)) if epoch_loss else 0.0
    print(f"Test loss:  {avg_loss}\n")
    writer.add_scalar("loss / test", avg_loss, epoch)

    for name, vals in batch_metrics_list.items():
        m = float(np.mean(vals)) if vals else 0.0
        print(f"Test {name}: {m}\n")
        writer.add_scalar(f"{name} / test", m, epoch)


In [ ]:
def train(
    n_epochs: int,
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: torch.nn.Module,
    writer: SummaryWriter,
    device: torch.device,
    transformers: bool = False,
) -> None:
    """
    Training loop.
    """

    for epoch in range(n_epochs):

        print(f"Epoch [{epoch+1} / {n_epochs}]\n")

        train_epoch(
            model=model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            criterion=criterion,
            writer=writer,
            device=device,
            epoch=epoch,
            transformers=transformers,

        )
        evaluate_epoch(
            model=model,
            dataloader=test_dataloader,
            criterion=criterion,
            writer=writer,
            device=device,
            epoch=epoch,
            transformers=transformers,
        )

In [ ]:
batch_size = 64
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collator,
)

valid_dataloader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=batch_size,  # для корректных замеров метрик оставить batch_size=1
    shuffle=False, # для корректных замеров метрик оставить shuffle=False
    collate_fn=collator,
)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,  # для корректных замеров метрик оставить batch_size=1
    shuffle=False, # для корректных замеров метрик оставить shuffle=False
    collate_fn=collator,
)

emb_dim = 100
hidden_dim = 100
layers= 3
dropout = 0.15

model = BiLSTM(
    num_embeddings=len(token2idx),
    embedding_dim=emb_dim,
    hidden_size=hidden_dim,
    num_layers=layers,
    dropout=dropout,
    bidirectional=True,
    n_classes=len(label2idx),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)


train(
      n_epochs=5,
      model=model,
      train_dataloader=train_dataloader,
      test_dataloader=valid_dataloader,
      optimizer=optimizer,
      criterion=criterion,
      writer=writer,
      device=device,
      )

Epoch [1 / 5]



loop over train batches: 100%|██████████| 220/220 [00:08<00:00, 25.53it/s]


Train loss: 0.36955953606150366

Train accuracy: 0.8980656750428221

Train precision_micro: 0.8980656750428221

Train precision_macro: 0.5902183269529628

Train precision_weighted: 0.8671861677274498

Train recall_micro: 0.8980656750428221

Train recall_macro: 0.49607033191946937

Train recall_weighted: 0.8980656750428221

Train f1_micro: 0.8980656750428221

Train f1_macro: 0.5180079938696539

Train f1_weighted: 0.8769940769798931



loop over test batches: 100%|██████████| 51/51 [00:01<00:00, 34.60it/s]


Test loss:  0.1870575859759222

Test accuracy: 0.946299243009328

Test precision_micro: 0.946299243009328

Test precision_macro: 0.7739174668296479

Test precision_weighted: 0.9449317197563872

Test recall_micro: 0.946299243009328

Test recall_macro: 0.7033025852246436

Test recall_weighted: 0.946299243009328

Test f1_micro: 0.946299243009328

Test f1_macro: 0.7207902020177969

Test f1_weighted: 0.9429897593213502

Epoch [2 / 5]



...

Test loss:  0.1714966848764258

Test accuracy: 0.9592031738749255

Test precision_micro: 0.9592031738749255

Test precision_macro: 0.8130885571613871

Test precision_weighted: 0.9598905311590155

Test recall_micro: 0.9592031738749255

Test recall_macro: 0.7912481492308562

Test recall_weighted: 0.9592031738749255

Test f1_micro: 0.9592031738749255

Test f1_macro: 0.7884472940427569

Test f1_weighted: 0.9578844342588516

Epoch [5 / 5]



loop over train batches: 100%|██████████| 220/220 [00:08<00:00, 25.00it/s]


Train loss: 0.03883276400579647

Train accuracy: 0.9882071829645964

Train precision_micro: 0.9882071829645964

Train precision_macro: 0.950518973012737

Train precision_weighted: 0.9886333285177958

Train recall_micro: 0.9882071829645964

Train recall_macro: 0.9400673781792571

Train recall_weighted: 0.9882071829645964

Train f1_micro: 0.9882071829645964

Train f1_macro: 0.941336951717411

Train f1_weighted: 0.9880766021572703



loop over test batches: 100%|██████████| 51/51 [00:01<00:00, 30.83it/s]

Test loss:  0.16797632092664785

Test accuracy: 0.958748738336136

Test precision_micro: 0.958748738336136

Test precision_macro: 0.8066601571452168

Test precision_weighted: 0.9593179416841223

Test recall_micro: 0.958748738336136

Test recall_macro: 0.7771224927748385

Test recall_weighted: 0.958748738336136

Test f1_micro: 0.958748738336136

Test f1_macro: 0.7780398838821653

Test f1_weighted: 0.9572712065616529



# TransformersDataset

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_name = "distilbert-base-cased"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
class TransformersDataset(Dataset):
    def __init__(
        self,
        token_seq: List[List[str]],
        label_seq: List[List[str]],
        token2idx: Dict[str, int],
        label2idx: Dict[str, int],
    ):
        self.token2idx = token2idx
        self.label2idx = label2idx
        self.token_seq = [self.process_tokens(tokens, token2idx) for tokens in token_seq]
        self.label_seq = [self.process_labels(labels, label2idx) for labels in label_seq]
        token_set = {v: k for k, v in self.token2idx.items()}
        label_set = {v: k for k, v in self.label2idx.items()}
        # Сохраняем списки как атрибуты, чтобы использовать в __getitem__
        self.words_from_token = list(map(lambda x: token_set[x], self.token_seq[0]))
        self.words_from_label = list(map(lambda x: label_set[x], self.label_seq[0]))


    def __len__(self):
        return len(self.token_seq)

    def __getitem__(self, idx: int):
        # Возвращаем кортеж с словами и индексами меток
        # Текущий код возвращает только для idx=0, если хотите для всех — их тоже надо хранить

        # Например, для демонстрации только 0 индекса:
        if idx == 0:
            return self.words_from_token, self.label_seq[idx]
        else:
            # Чтобы для других индексов вернуть слова, нужно декодировать обратным словарём:
            token_set = {v: k for k, v in self.token2idx.items()}
            label_set = {v: k for k, v in self.label2idx.items()}
            words_token = list(map(lambda x: token_set[x], self.token_seq[idx]))
            words_label = list(map(lambda x: label_set[x], self.label_seq[idx]))
            return words_token, words_label #self.label2idx[words_label[idx]]

    @staticmethod
    def process_tokens(tokens: List[str], token2idx: Dict[str, int], unk: str = "<UNK>") -> List[int]:
        return [token2idx.get(token, token2idx[unk]) for token in tokens]

    @staticmethod
    def process_labels(labels: List[str], label2idx: Dict[str, int]) -> List[int]:
        return [label2idx[label] for label in labels]


In [ ]:
token2idx = get_token2idx(token2cnt, min_count=2)

In [ ]:
label2idx = get_label2idx(label_set)

In [ ]:
train_dataset[9]

(tensor([126,  94, 127,   5, 128, 129,  88, 111,  14,   2,  37, 130,  40,  41,
          67,   1, 101, 108, 131,  67, 132,  81,  61,  84,  54, 133, 134, 135,
          54, 136,  73, 137, 138,   5, 100, 108,   9]),
 tensor([0, 4, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

In [ ]:
train_dataset = TransformersDataset(
    token_seq=train_token_seq,
    label_seq=train_label_seq,
    token2idx=token2idx,
    label2idx=label2idx,
)

valid_dataset = TransformersDataset(
    token_seq=valid_token_seq,
    label_seq=valid_label_seq,
    token2idx=token2idx,
    label2idx=label2idx,
)

test_dataset = TransformersDataset(
    token_seq=test_token_seq,
    label_seq=test_label_seq,
    token2idx=token2idx,
    label2idx=label2idx,
)


In [ ]:
label2idx

{'O': 0,
 'B-LOC': 1,
 'B-MISC': 2,
 'B-ORG': 3,
 'B-PER': 4,
 'I-LOC': 5,
 'I-MISC': 6,
 'I-ORG': 7,
 'I-PER': 8}

In [ ]:
train_dataset[0]

(['eu', '<UNK>', 'german', 'call', 'to', 'boycott', 'british', 'lamb', '.'],
 [3, 0, 2, 0, 0, 0, 2, 0, 0])

In [ ]:
valid_dataset[0]

(['cricket',
  '-',
  'leicestershire',
  'take',
  'over',
  'at',
  'top',
  'after',
  'innings',
  'victory',
  '.'],
 [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
test_dataset[0]

(['soccer',
  '-',
  'japan',
  'get',
  'lucky',
  'win',
  ',',
  'china',
  'in',
  'surprise',
  'defeat',
  '.'],
 [0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0])

# TransformersCollator

In [ ]:
import torch
import numpy as np
from typing import List, Tuple, Dict, Any, Optional
from transformers import PreTrainedTokenizer
from transformers.tokenization_utils_base import BatchEncoding

class TransformersCollator:
    def __init__(self, tokenizer: PreTrainedTokenizer,
                 tokenizer_kwargs: Dict[str, Any],
                 label_padding_value: int,
                 label2idx: Optional[Dict[str, int]] = None,):
        self.tokenizer = tokenizer
        self.tokenizer_kwargs = tokenizer_kwargs
        self.label_padding_value = label_padding_value
        self.label2idx = label2idx

    def __call__(self, batch: List[Tuple[List[str], List[int]]]) -> Tuple[Dict[str, torch.Tensor], torch.Tensor]:
        inputs, labels = zip(*batch)

        tokenizer_kwargs = dict(self.tokenizer_kwargs)
        for key in ['is_split_into_words', 'return_offsets_mapping', 'padding', 'return_tensors']:
            tokenizer_kwargs.pop(key, None)

        tokens : BatchEncoding = self.tokenizer(
            list(inputs),
            is_split_into_words=True,
            padding=True,
            return_offsets_mapping=True,
            return_tensors="pt",
            **self.tokenizer_kwargs,
         )

        labels_tensor = self.encode_labels(tokens, labels, self.label_padding_value, self.label2idx)

        if "offset_mapping" in tokens:
            tokens.pop("offset_mapping")
        return tokens, labels_tensor

    @staticmethod
    def encode_labels(tokens: BatchEncoding, labels: List[List[int]], label_padding_value: int, label2idx) -> torch.LongTensor:
        encoded_labels = []
        for doc_labels, doc_offset in zip(labels, tokens["offset_mapping"]):
            if len(doc_labels) > 0 and isinstance(doc_labels[0], str):
                if label2idx is None:
                    raise ValueError("Переданы строковые метки, но label2idx не задан.")
                doc_labels = [label2idx[l] for l in doc_labels]

            doc_enc_labels = np.ones(len(doc_offset), dtype=int) * label_padding_value
            arr = np.array(doc_offset)
            mask = (arr[:, 0] == 0) & (arr[:, 1] != 0)

            if mask.sum() != len(doc_labels):
                raise ValueError(
                    f"Несоответствие длин: {mask.sum()} позиций для меток, "
                    f"но меток {len(doc_labels)}."
                )

            doc_enc_labels[mask] = np.asarray(doc_labels, dtype=int)
            encoded_labels.append(doc_enc_labels.tolist())

        return torch.LongTensor(encoded_labels)


In [ ]:
tokenizer_kwargs = {
    "is_split_into_words":    True,
    "return_offsets_mapping": True,
    "padding":                True,
    "truncation":             True,
    "max_length":             512,
    "return_tensors":         "pt",
}

collator = TransformersCollator(
    tokenizer=tokenizer,
    tokenizer_kwargs=tokenizer_kwargs,
    label_padding_value=-100,
    label2idx=label2idx
)

In [ ]:
train_dataset[10]

(['spanish',
  'farm',
  'minister',
  '<UNK>',
  'de',
  '<UNK>',
  'had',
  'earlier',
  'accused',
  'fischler',
  'at',
  'an',
  'eu',
  'farm',
  'ministers',
  "'",
  'meeting',
  'of',
  'causing',
  '<UNK>',
  'alarm',
  'through',
  '"',
  'dangerous',
  '<UNK>',
  '.',
  '"'],
 ['B-MISC',
  'O',
  'O',
  'B-PER',
  'I-PER',
  'I-PER',
  'O',
  'O',
  'O',
  'B-PER',
  'O',
  'O',
  'B-ORG',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'])

In [ ]:
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collator,
)
valid_dataloader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=2,
    shuffle=False, # для корректных замеров метрик оставить shuffle=False
    collate_fn=collator,
)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False, # для корректных замеров метрик оставить shuffle=False
    collate_fn=collator,
)

In [ ]:
tokens, labels = next(iter(train_dataloader))

tokens = tokens.to(device)
labels = labels.to(device)

In [ ]:
tokens.items()

ItemsView({'input_ids': tensor([[  101,   174,  1358,   133,  7414,  2428,   135,   176, 14170,  1840,
          1106, 21423,  9304, 10721,  1324,  2495, 12913,   119,   102],
        [  101, 11109,  1200,  1602,  6715,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
       device='cuda:0')})

In [ ]:
train_tokens, train_labels = next(iter(
    torch.utils.data.DataLoader(
        train_dataset,
        batch_size=2,
        shuffle=False,
        collate_fn=collator,
    )
))

valid_tokens, valid_labels = next(iter(
    torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=2,
        shuffle=False,
        collate_fn=collator,
    )
))

test_tokens, test_labels = next(iter(
    torch.utils.data.DataLoader(
        test_dataset,
        batch_size=2,
        shuffle=False,
        collate_fn=collator,
    )
))

print("Тесты пройдены!")

In [ ]:
tokens, labels = next(iter(train_dataloader))

tokens = tokens.to(device)
labels = labels.to(device)

# BERT

In [ ]:
from transformers import AutoModelForTokenClassification

In [ ]:
from transformers import DistilBertConfig, DistilBertForTokenClassification

config = DistilBertConfig.from_pretrained(model_name, num_labels=len(label2idx))
model = DistilBertForTokenClassification(config)


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model.to(device)

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=label_padding_value)

In [ ]:
train(
      n_epochs=5,
      model=model,
      train_dataloader=train_dataloader,
      test_dataloader=valid_dataloader,
      optimizer=optimizer,
      criterion=criterion,
      writer=writer,
      device=device,
      transformers=True,
      )

Epoch [1 / 5]



loop over train batches: 100%|██████████| 7021/7021 [06:40<00:00, 17.54it/s]


Train loss: 0.4691463461424618

Train accuracy: 0.41460700491864594

Train precision_micro: 0.41460700491864594

Train precision_macro: 0.19900008380248177

Train precision_weighted: 0.22174801050765666

Train recall_micro: 0.41460700491864594

Train recall_macro: 0.36194499929743507

Train recall_weighted: 0.41460700491864594

Train f1_micro: 0.41460700491864594

Train f1_macro: 0.23668474128498132

Train f1_weighted: 0.27883333145436723



loop over test batches: 100%|██████████| 1625/1625 [00:41<00:00, 39.15it/s]


Test loss:  0.43965336751937867

Test accuracy: 0.40334011995606306

Test precision_micro: 0.40334011995606306

Test precision_macro: 0.19925475208233429

Test precision_weighted: 0.2106809939637126

Test recall_micro: 0.40334011995606306

Test recall_macro: 0.35178835500884514

Test recall_weighted: 0.40334011995606306

Test f1_micro: 0.40334011995606306

Test f1_macro: 0.23451911347494905

Test f1_weighted: 0.26728024008160195

Epoch [2 / 5]



...

Test loss:  0.30766428075367225

Test accuracy: 0.4211978543352181

Test precision_micro: 0.4211978543352181

Test precision_macro: 0.24094189672341956

Test precision_weighted: 0.2274419442449317

Test recall_micro: 0.4211978543352181

Test recall_macro: 0.4137420687361948

Test recall_weighted: 0.4211978543352181

Test f1_micro: 0.4211978543352181

Test f1_macro: 0.2833445440775911

Test f1_weighted: 0.28572500745235857

Epoch [5 / 5]



loop over train batches: 100%|██████████| 7021/7021 [06:39<00:00, 17.59it/s]


Train loss: 0.1491577940548534

Train accuracy: 0.4509262428102978

Train precision_micro: 0.4509262428102978

Train precision_macro: 0.28432831039544937

Train precision_weighted: 0.2588538185475649

Train recall_micro: 0.4509262428102978

Train recall_macro: 0.4941754337202013

Train recall_weighted: 0.4509262428102978

Train f1_micro: 0.4509262428102978

Train f1_macro: 0.33811012586079897

Train f1_weighted: 0.31943603225169126



loop over test batches: 100%|██████████| 1625/1625 [00:44<00:00, 36.82it/s]

Test loss:  0.291427079824883

Test accuracy: 0.4234025070724383

Test precision_micro: 0.4234025070724383

Test precision_macro: 0.26130198723801845

Test precision_weighted: 0.22837176117037072

Test recall_micro: 0.4234025070724383

Test recall_macro: 0.4465735223297188

Test recall_weighted: 0.4234025070724383

Test f1_micro: 0.4234025070724383

Test f1_macro: 0.305980135328106

Test f1_weighted: 0.28663770467623206



# BiLSTMAttention

In [ ]:
import torch

class BiLSTMAttn(torch.nn.Module):
    """
    Bidirectional LSTM architecture with Multihead Attention.
    """

    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool,
        n_classes: int,
        num_heads: int,
    ):
        super().__init__()

        self.embedding = torch.nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.rnn = torch.nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
            batch_first=True
        )
        self.embed_dim = hidden_size * 2 if bidirectional else hidden_size
        self.attn = torch.nn.MultiheadAttention(embed_dim=self.embed_dim, num_heads=num_heads)
        self.head = torch.nn.Linear(in_features=self.embed_dim, out_features=n_classes)


    def forward(self, tokens: torch.LongTensor) -> torch.Tensor:
        embed = self.embedding(tokens)

        length = (tokens != 0).sum(dim=1).detach().cpu()
        packed_embed = torch.nn.utils.rnn.pack_padded_sequence(
            embed, length, batch_first=True, enforce_sorted=False
        )

        packed_rnn_output, _ = self.rnn(packed_embed)
        rnn_output, _ = torch.nn.utils.rnn.pad_packed_sequence(
            packed_rnn_output, batch_first=True
        )  # Shape: (batch_size, seq_len, embed_dim)

        # MultiheadAttention expects input shape: (seq_len, batch_size, embed_dim), транспонируем
        rnn_output = rnn_output.transpose(0, 1)
        attn_output, _ = self.attn(rnn_output, rnn_output, rnn_output)  # self-attention

        # Возвращаем обратно (batch_size, seq_len, embed_dim)
        attn_output = attn_output.transpose(0, 1)

        logits = self.head(attn_output)  # (batch_size, seq_len, n_classes)
        return logits.transpose(1, 2)  # (batch_size, n_classes, seq_len)


In [ ]:
model = BiLSTMAttn(
    num_embeddings=len(token2idx),
    embedding_dim=emb_dim,
    hidden_size=hidden_dim,
    num_layers=layers,
    dropout=dropout,
    bidirectional=True,
    n_classes=len(label2idx),
    num_heads = 5,
).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)

In [ ]:
model

BiLSTMAttn(
  (embedding): Embedding(10951, 100)
  (rnn): LSTM(100, 100, num_layers=3, batch_first=True, dropout=0.15, bidirectional=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=200, out_features=200, bias=True)
  )
  (head): Linear(in_features=200, out_features=9, bias=True)
)

In [ ]:
def _move_batch_to_device(tokens, labels, device, transformers: bool):
    """Move batch to device; для BiLSTM берём именно idxы токенов."""
    if transformers:
        tokens = {k: v.to(device) for k, v in tokens.items()}
    # else:
    #     if isinstance(tokens, dict):        # если пришёл BatchEncoding
    #         tokens = tokens["input_ids"]    # BiLSTM ждёт [B, T] индексы
    #     tokens = tokens.to(device)
    # labels = labels.to(device).long()
    else:
        if isinstance(tokens, dict):  # если пришёл BatchEncoding
            tokens = tokens["input_ids"].to(device)  # BiLSTM ждёт [B, T] индексы
        else:
            tokens = tokens.to(device)
        labels = labels.to(device).long()
    return tokens, labels


def _to_btc_for_metrics(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Приводит logits к [B, T, C] (для метрик).
    Допускает вход [B,T,C] или [B,C,T]; сверяет с T = labels.size(1).
    """
    T = labels.size(1)
    if logits.dim() != 3:
        raise ValueError(f"Expected 3D logits, got {logits.shape}")
    if logits.shape[1] == T:         # [B, T, C]
        return logits
    if logits.shape[2] == T:         # [B, C, T]
        return logits.transpose(1, 2)
    raise ValueError(f"Token length T={T} not found in logits shape {logits.shape}")

def _to_bct_for_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Приводит logits к [B, C, T] (для CrossEntropyLoss).
    """
    T = labels.size(1)
    if logits.dim() != 3:
        raise ValueError(f"Expected 3D logits, got {logits.shape}")
    if logits.shape[2] == T:         # [B, C, T]
        return logits
    if logits.shape[1] == T:         # [B, T, C]
        return logits.transpose(1, 2)
    raise ValueError(f"Token length T={T} not found in logits shape {logits.shape}")




In [ ]:
def _move_batch_to_device(tokens, labels, device, transformers: bool):
    """Move batch to device; для BiLSTM берём именно idxы токенов."""
    if transformers:
        # Прямое перемещение всех ключей BatchEncoding на устройство
        tokens = {k: v.to(device) for k, v in tokens.items()}
        labels = labels.to(device).long()
    else:
        # Строгая проверка и конвертация для не-transformer моделей
        if isinstance(tokens, dict):
            tokens = tokens['input_ids'].to(device)
        else:
            tokens = tokens.to(device)
        labels = labels.to(device).long()

    # Добавляем проверку типа данных
    if not isinstance(tokens, torch.Tensor):
        raise ValueError(f"Неверный тип данных для tokens: {type(tokens)}")

    return tokens, labels

def train_epoch(model, dataloader, optimizer, criterion, writer, device, epoch, transformers: bool = False):
    model.train()
    epoch_loss = []
    batch_metrics_list = defaultdict(list)

    for i, (tokens, labels) in tqdm(enumerate(dataloader), total=len(dataloader)):
        try:
            optimizer.zero_grad()
            tokens, labels = _move_batch_to_device(tokens, labels, device, transformers)

            if transformers:
                out = model(**tokens, labels=labels)
                loss = out.loss
                logits_for_metrics = _to_btc_for_metrics(out.logits, labels)
            else:
                logits = model(tokens)
                logits_for_loss = _to_bct_for_loss(logits, labels)
                loss = criterion(logits_for_loss, labels)
                logits_for_metrics = _to_btc_for_metrics(logits, labels)

            loss.backward()
            optimizer.step()

            epoch_loss.append(loss.item())
            writer.add_scalar("batch loss / train", loss.item(), epoch * len(dataloader) + i)

            with torch.no_grad():
                batch_metrics = compute_metrics(outputs=logits_for_metrics, labels=labels, ignore_index=IGNORE_INDEX)
                for name, val in batch_metrics.items():
                    batch_metrics_list[name].append(val)
                    writer.add_scalar(f"batch {name} / train", val, epoch * len(dataloader) + i)

        except Exception as e:
            print(f"Ошибка при обработке батча {i}: {str(e)}")
            continue

    if epoch_loss:
        avg_loss = float(np.mean(epoch_loss))
        print(f"Train loss: {avg_loss}\n")
        writer.add_scalar("loss / train", avg_loss, epoch)

        for name, vals in batch_metrics_list.items():
            if vals:
                m = float(np.mean(vals))
                print(f"Train {name}: {m}\n")
                writer.add_scalar(f"{name} / train", m, epoch)

In [ ]:
train(
    n_epochs=5,
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=valid_dataloader,
    optimizer=optimizer,
    criterion=criterion,
    writer=writer,
    device=device,
    transformers=False  #  False
)


Epoch [1 / 5]



100%|██████████| 220/220 [00:10<00:00, 21.30it/s]


Train loss: 1.1306429386138916

Train accuracy: 0.7742250547341766

Train precision_micro: 0.7742250547341766

Train precision_macro: 0.09004783635166515

Train precision_weighted: 0.667382895252159

Train recall_micro: 0.7742250547341766

Train recall_macro: 0.11243382018569649

Train recall_weighted: 0.7742250547341766

Train f1_micro: 0.7742250547341766

Train f1_macro: 0.09635595961350646

Train f1_weighted: 0.7111643329691827



loop over test batches: 100%|██████████| 51/51 [00:01<00:00, 42.91it/s]


Test loss:  0.8513953785101572

Test accuracy: 0.8103933908978479

Test precision_micro: 0.8103933908978479

Test precision_macro: 0.09838736771385434

Test precision_weighted: 0.6634651942541107

Test recall_micro: 0.8103933908978479

Test recall_macro: 0.12276688453159039

Test recall_weighted: 0.8103933908978479

Test f1_micro: 0.8103933908978479

Test f1_macro: 0.10888723828015169

Test f1_weighted: 0.7278683511092172

Epoch [2 / 5]



...

Test loss:  0.7379042512061549

Test accuracy: 0.8109775702709546

Test precision_micro: 0.8109775702709546

Test precision_macro: 0.10258484685781155

Test precision_weighted: 0.6662357017761222

Test recall_micro: 0.8109775702709546

Test recall_macro: 0.12361575622445185

Test recall_weighted: 0.8109775702709546

Test f1_micro: 0.8109775702709546

Test f1_macro: 0.11019399108732092

Test f1_weighted: 0.7289857301607418

Epoch [5 / 5]



100%|██████████| 220/220 [00:09<00:00, 22.52it/s]


Train loss: 0.6489690718325701

Train accuracy: 0.8354305110472222

Train precision_micro: 0.8354305110472222

Train precision_macro: 0.16516924773645808

Train precision_weighted: 0.7175589301131035

Train recall_micro: 0.8354305110472222

Train recall_macro: 0.12508483237942145

Train recall_weighted: 0.8354305110472222

Train f1_micro: 0.8354305110472222

Train f1_macro: 0.12296027049487268

Train f1_weighted: 0.7641998011280525



loop over test batches: 100%|██████████| 51/51 [00:01<00:00, 41.89it/s]

Test loss:  0.6667411426703135

Test accuracy: 0.8152171621422903

Test precision_micro: 0.8152171621422903

Test precision_macro: 0.14970193736080603

Test precision_weighted: 0.6952523692009658

Test recall_micro: 0.8152171621422903

Test recall_macro: 0.1334798069928513

Test recall_weighted: 0.8152171621422903

Test f1_micro: 0.8152171621422903

Test f1_macro: 0.12493989912685993

Test f1_weighted: 0.7379688028523784

